In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
files = sorted(glob.glob('/home/ulyanov/data/stereo/*'))
files

['/home/ulyanov/data/stereo/hmi.M_45s.20240213_233000_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/stereo/hmi.M_45s.20250313_175530_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/stereo/hmi.V_45s.20240213_233000_TAI.2.Dopplergram.fits',
 '/home/ulyanov/data/stereo/hmi.V_45s.20250313_175530_TAI.2.Dopplergram.fits',
 '/home/ulyanov/data/stereo/solo_L2_phi-fdt-blos_20240213T233009_V202602220902_0442130501.fits.gz',
 '/home/ulyanov/data/stereo/solo_L2_phi-fdt-blos_20250313T180009_V202602220258_0543130503.fits.gz',
 '/home/ulyanov/data/stereo/solo_L2_phi-fdt-vlos_20240213T233009_V202602220902_0442130501.fits.gz',
 '/home/ulyanov/data/stereo/solo_L2_phi-fdt-vlos_20250313T180009_V202602220258_0543130503.fits.gz']

In [3]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [19]:
file_hmi = files[1]
file_fdt = files[5]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

did = int(file_fdt.split('_')[-1].split('.')[0])

data_fdt = undistort(data_fdt, header_fdt, xd, yd) * 1.3
data_hmi = rebin(data_hmi, 8, update_header=header_hmi)

view_fdt = View.from_header(header_fdt).update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=header_fdt['CROTA'] + 0.3, rsun=ru_sun[dids == did][0])
view_hmi = View.from_header(header_hmi)


view_new = view_fdt.update(crota=0)#, crlt=0)
transform_fdt = (view_new.to_carrington() -
                 view_fdt.to_carrington(mu_thr=0.2))


grid, alpha = transform_fdt(view_new.grid())
data_fdt = bilinear(data_fdt, *grid) * alpha

#view_new = view_fdt.update(crlt=view_hmi.crlt, crota=0)
transform_hmi = (view_new.to_carrington() -
                 view_hmi.to_carrington(mu_thr=0.2))

grid, alpha = transform_hmi(view_new.grid())
data_hmi = bilinear(data_hmi, *grid) * alpha

In [20]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi, cmap='seismic', vmin=-200, vmax=200)
plt.tight_layout()

In [21]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt - data_hmi, cmap='seismic', vmin=-200, vmax=200)
plt.tight_layout()

In [15]:
transform = view_hmi.to_carrington(origin='helioprojective') - view_fdt.to_carrington(origin='helioprojective')

e1 = (0,0,1)
e2 = transform(e1)[0]

e1 = np.array(e1)
e2 = np.array(e2)

q = np.sum(e1 * e2)
delta = np.sqrt(1 - q ** 2)

e2_ = (e2 - e1 * q) / delta

v1 = data_fdt
v2 = data_hmi

V11 = np.sqrt((v1 ** 2 + v2 ** 2 - 2 * v1 * v2 * q)) / delta

Vq = np.sqrt(V11 ** 2 - v1 ** 2)

In [16]:
plt.figure(figsize=(10,10))
plt.imshow(V11, cmap='jet', vmin=0, vmax=500)
plt.tight_layout()

In [10]:
plt.figure(figsize=(10,10))
plt.imshow(Vq, cmap='jet', vmin=0, vmax=500)
plt.tight_layout()

In [17]:
plt.figure(figsize=(10,10))
plt.imshow(np.abs(v1), cmap='jet', vmin=0, vmax=200)
plt.tight_layout()

In [21]:
view_fdt.tdel / 60

4.3949676080099005